In [ ]:
from tqdm import tqdm
import torch

from gymnasium_env.wrappers import MultipleColorsEncoding

from src.agents.qnetwork.training_algorithtms import TrainDQN, MyAlgo
from src.agents.qnetwork.archis import UnicolorArch, ColorfulArch
from src.agents.qnetwork.qnetwork_agent import QNetworkAgent

from src.simulator import BlokusGameSimulator

from src.agents import RandomAgent, HeuristicAgent, MiniMaxAgent

from src.agents.heuristic.heuristics import greedy, min_his_expanders, max_my_expanders, min_his_possible_actions, min_actions_after_size

In [ ]:
BOARD_SIZE = 10
PLAYER_TURN = 1
DEVICE = (
    "cuda" if torch.cuda.is_available() else 
    "mps" if torch.backends.mps.is_available() else 
    "cpu"
)
USE_WANDB = False

In [ ]:
Random = RandomAgent()
MinHisExpanders = HeuristicAgent(func=min_his_expanders, name="MinHisExpanders", board_size=BOARD_SIZE)
MaxMyExpanders = HeuristicAgent(func=max_my_expanders, name="MaxMyExpanders", board_size=BOARD_SIZE)
Greedy = HeuristicAgent(func=greedy, name="Greedy", board_size=BOARD_SIZE)
MinHisActions = HeuristicAgent(func=min_his_possible_actions, name="MinHisActions", board_size=BOARD_SIZE)
MinActionsAfterSize = HeuristicAgent(func=min_actions_after_size, name="MinActionsAfterSize", board_size=BOARD_SIZE)

In [ ]:
agent = QNetworkAgent(
    board_size=BOARD_SIZE,
    model_class=ColorfulArch,
    device=DEVICE,
    model_folder="20250504_214208"
)

opponent_agent = Random

wrappers = [MultipleColorsEncoding] if agent.model_class == ColorfulArch else []

trainer = TrainDQN(
    agent=agent,
    device=DEVICE,
    player_turn=PLAYER_TURN,
    wrappers=wrappers,
    batch_size=64,
    lr=0.0001,
    gamma=0.99,
    epsilon=1,
    min_epsilon=0.01,
    epsilon_decay=0.99,
    target_update_freq=50,
    buffer_size=1000,
    use_wandb=USE_WANDB,
)

simulator = BlokusGameSimulator(
    board_size=BOARD_SIZE,
    num_players=2,
    wrappers=wrappers,
)

In [ ]:
trainer.collect_trajectories(max_steps=1000, pbar=True, hidden_agent=opponent_agent)

In [ ]:
pbar = tqdm(range(1000), desc="Training Progress")

for i in pbar:
    trainer.collect_trajectories(max_steps=64, pbar=False)
    trainer.optimize_model()
    trainer.update_epsilon()
    if i % 50 == 0:
        log_dict = simulator.test(
            agent1=agent,
            agent2=Random,
            testing_agent_id=PLAYER_TURN,
            num_episodes=100
        )
        win_rate = log_dict["win_rate"]
        tie_rate = log_dict["tie_rate"]
        lose_rate = log_dict["lose_rate"]
        diff_points = log_dict["diff_points"]
        pbar.set_description(
            f"[{i}] => {len(trainer.replay_buffer)} ε: {trainer.epsilon:.4f}, Win rate: {win_rate:.2f}, Tie rate: {tie_rate:.2f}, Lose rate: {lose_rate:.2f}"
        )
        if USE_WANDB:
            trainer.run.log({"win_rate": win_rate, "tie_rate": tie_rate, "lose_rate": lose_rate, "diff_points": diff_points}, step=trainer.step_count)

In [ ]:
trainer.save_model()

In [ ]:
simulator = BlokusGameSimulator(
    board_size=BOARD_SIZE,
    num_players=2,
    wrappers=wrappers,
)

In [ ]:
win_rate, tie_rate, lose_rate = simulator.test(
    agent1=agent,
    agent2=MinActionsAfterSize,
    testing_agent_id=PLAYER_TURN,
    num_episodes=100,
    pbar=True
)
print(f"Win rate: {win_rate:.2f}, Tie rate: {tie_rate:.2f}, Lose rate: {lose_rate:.2f}.")